# CardioGuard Phase 2 (Colab)
Run: **Runtime > Run all**. The first cell installs wfdb; the MIT-BIH download (~100 MB) happens automatically.

In [ ]:
!pip install -q wfdb scikit-learn pandas matplotlib

In [ ]:
# ---- features.py ----
"""Turn a beat-time series into fixed-length window features + labels.

IMPORTANT: features use ONLY beat timing (RR intervals), the same signal the phone gets from a
watch. Labels use the cardiologist beat annotations and are NEVER used as features.
"""
import numpy as np
import pandas as pd

FEATURES = ["mean_hr", "sdnn", "rmssd", "pnn50", "max_jump",
            "min_ratio", "max_ratio", "premature_frac"]

# Annotation symbols that mark rhythm-relevant ectopy (PVC, PAC, aberrant, junctional, escape...).
# Bundle-branch blocks (L, R) and paced beats are morphology/pacing, not timing anomalies -> excluded.
ECTOPIC = ("A", "a", "J", "S", "V", "F", "e", "j", "E", "r")


def window_features(beat_times, symbols, window_s=30.0, min_beats=10,
                    min_ectopic=2, ectopic=ECTOPIC) -> pd.DataFrame:
    """Non-overlapping windows. Label = 1 if the window holds >= min_ectopic ectopic beats."""
    t = np.asarray(beat_times, dtype=float)
    sym = np.asarray(symbols)
    is_ect = np.isin(sym, list(ectopic))
    rows = []
    for w0 in np.arange(t[0], t[-1] - window_s + 1e-9, window_s):
        m = (t >= w0) & (t < w0 + window_s)
        if m.sum() < min_beats:
            continue
        rr = np.diff(t[m]) * 1000.0          # ms
        d = np.diff(rr)
        med = np.median(rr)
        rows.append({
            "start": w0,
            "mean_hr": 60000.0 / rr.mean(),
            "sdnn": rr.std(ddof=1),
            "rmssd": np.sqrt(np.mean(d ** 2)),
            "pnn50": np.mean(np.abs(d) > 50),
            "max_jump": np.max(np.abs(d)) / med,
            "min_ratio": rr.min() / med,
            "max_ratio": rr.max() / med,
            "premature_frac": np.mean(rr < 0.8 * med),
            "n_ectopic": int(is_ect[m].sum()),
        })
    df = pd.DataFrame(rows)
    if len(df):
        df["label"] = (df["n_ectopic"] >= min_ectopic).astype(int)
    return df


In [ ]:
# ---- data.py ----
"""Loading MIT-BIH (real) or synthetic beat data. Both return {record: (beat_times_s, symbols)}."""
import os
import numpy as np

# 48 records; 102, 104, 107, 217 are paced (standard practice is to exclude them).
MITDB_RECORDS = [str(r) for r in
                 list(range(100, 110)) + list(range(111, 120)) + [121, 122, 123, 124] +
                 list(range(200, 204)) + [205, 207, 208, 209, 210, 212, 213, 214, 215,
                                          217, 219, 220, 221, 222, 223, 228, 230, 231,
                                          232, 233, 234]]
PACED = {"102", "104", "107", "217"}
BEAT_SYMBOLS = set("NLRBAaJSVrFejnE/fQ?")


def load_mitdb(data_dir="data/mitdb", records=None, exclude_paced=True):
    import wfdb  # imported lazily so synthetic mode works without it
    if not os.path.isdir(data_dir) or not any(f.endswith(".atr") for f in os.listdir(data_dir)):
        wfdb.dl_database("mitdb", data_dir)  # downloads ~100 MB once
    out = {}
    for rec in (records or MITDB_RECORDS):
        if exclude_paced and rec in PACED:
            continue
        ann = wfdb.rdann(os.path.join(data_dir, rec), "atr")
        keep = np.array([s in BEAT_SYMBOLS for s in ann.symbol])
        times = ann.sample[keep] / float(ann.fs)
        syms = np.array(ann.symbol)[keep]
        out[rec] = (times, syms)
    return out


def synthetic_records(n_records=20, minutes=30, seed=0):
    """Fake beat series with sinus rhythm + respiratory modulation + bursts of PVCs.
    Only for testing the pipeline offline. NOT a substitute for real results."""
    rng = np.random.default_rng(seed)
    out = {}
    for i in range(n_records):
        base_rr = 60.0 / rng.uniform(55, 95)
        resp_amp = rng.uniform(0.01, 0.05)
        total = minutes * 60
        episodes = []
        for _ in range(rng.integers(1, 5)):
            s = rng.uniform(600, total - 200)     # keep first 10 min clean-ish (baseline)
            episodes.append((s, s + rng.uniform(40, 150)))
        t, times, syms = 0.0, [], []
        pending_comp = False
        while t < total:
            rr = base_rr * (1 + resp_amp * np.sin(2 * np.pi * t / 4.0)) + rng.normal(0, 0.012)
            in_ep = any(a <= t <= b for a, b in episodes)
            sym = "N"
            if pending_comp:
                rr *= 1.4
                pending_comp = False
            elif in_ep and rng.random() < 0.18:
                rr *= 0.6
                sym = "V"
                pending_comp = True
            t += max(rr, 0.25)
            times.append(t)
            syms.append(sym)
        out[f"syn{i:02d}"] = (np.array(times), np.array(syms))
    return out


In [ ]:
# ---- model.py ----
"""Detectors + leave-one-record-out evaluation.

Four detectors are compared on each held-out "user" (record):
  fixed_range     generic 60-100 bpm rule (the thing the app is supposed to beat)
  personal_z      only the user's own baseline (median/MAD z-scores), no population data
  population_if   Isolation Forest on absolute features, trained on OTHER records' normal windows
  personalized_if Isolation Forest on baseline-relative features, trained on other records' normal
                  windows PLUS this user's own (unlabeled) baseline windows  <- proposed model
Baseline = first N windows of each record (calibration period). Only later windows are scored.
"""
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve


DETECTORS = ["fixed_range", "personal_z", "population_if", "personalized_if"]


def baseline_stats(df, n_base):
    base = df.iloc[:n_base][FEATURES]
    med = base.median()
    mad = (base - med).abs().median()
    scale = 1.4826 * mad + 1e-3 + 0.05 * med.abs()   # floor so near-constant features don't explode
    return med, scale


def relativize(df, stats):
    med, scale = stats
    z = (df[FEATURES] - med) / scale
    return np.sign(z) * np.log1p(np.abs(z))          # compress heavy tails, keep sign


def _iforest(X, seed):
    scaler = StandardScaler().fit(X)
    m = IsolationForest(n_estimators=300, random_state=seed).fit(scaler.transform(X))
    return lambda Z: -m.score_samples(scaler.transform(Z))   # higher = more anomalous


def fit_population_model(records, n_base=20, seed=0):
    """Fit on all given records' normal windows (used for the exported/deployed model)."""
    Xs = [relativize(df, baseline_stats(df, n_base))[df["label"] == 0]
          for df in records.values() if len(df) > n_base]
    return _iforest(pd.concat(Xs).values, seed)


def leave_one_record_out(records, n_base=20, seed=0):
    """records: {name: window_df}. Returns long DataFrame of scores for post-baseline windows."""
    records = {k: v for k, v in records.items() if len(v) > n_base + 5}
    rel = {k: relativize(v, baseline_stats(v, n_base)) for k, v in records.items()}
    out = []
    for held, df in records.items():
        others = [k for k in records if k != held]
        test = df.iloc[n_base:]
        Xrel_test = rel[held].iloc[n_base:]
        own_base = rel[held].iloc[:n_base]

        # 1. fixed range
        hr = test["mean_hr"].values
        s_fixed = np.maximum.reduce([60 - hr, hr - 100, np.zeros_like(hr)])

        # 2. personal z (own baseline only)
        s_z = np.abs(Xrel_test.values).max(axis=1)

        # 3. population IF, absolute features
        Xabs = pd.concat([records[k][FEATURES][records[k]["label"] == 0] for k in others])
        s_pop = _iforest(Xabs.values, seed)(test[FEATURES].values)

        # 4. personalized IF (population normals in relative space + own baseline windows)
        Xrel = pd.concat([rel[k][records[k]["label"] == 0] for k in others] + [own_base])
        s_pers = _iforest(Xrel.values, seed)(Xrel_test.values)

        res = pd.DataFrame({"record": held, "start": test["start"].values,
                            "label": test["label"].values,
                            "fixed_range": s_fixed, "personal_z": s_z,
                            "population_if": s_pop, "personalized_if": s_pers})
        out.append(res)
    return pd.concat(out, ignore_index=True)


def _recall_at_fpr(y, s, fpr_target):
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.interp(fpr_target, fpr, tpr))


def summarize(results):
    """Pooled + per-record metrics."""
    pooled = []
    for d in DETECTORS:
        y, s = results["label"].values, results[d].values
        pooled.append({"detector": d,
                       "AUROC": roc_auc_score(y, s),
                       "AvgPrecision": average_precision_score(y, s),
                       "Recall@1%FPR": _recall_at_fpr(y, s, 0.01),
                       "Recall@5%FPR": _recall_at_fpr(y, s, 0.05)})
    per = []
    for rec, g in results.groupby("record"):
        if g["label"].nunique() < 2:
            continue
        per.append({"record": rec, "n_windows": len(g), "anom_rate": g["label"].mean(),
                    **{d: roc_auc_score(g["label"], g[d]) for d in DETECTORS}})
    return pd.DataFrame(pooled).set_index("detector"), pd.DataFrame(per).set_index("record")


def sustained_alerts(results, col, threshold, k=2):
    """Two-tier logic: 'urgent' only when >= k consecutive windows exceed threshold.
    Returns false alerts per hour (on normal-only stretches) and fraction of anomaly episodes caught."""
    fa, hours, caught, episodes = 0, 0.0, 0, 0
    for _, g in results.sort_values("start").groupby("record"):
        above = (g[col].values > threshold).astype(int)
        y = g["label"].values
        run = np.convolve(above, np.ones(k, dtype=int), mode="full")[:len(above)]
        urgent = run >= k
        hours += len(g) * 30 / 3600
        # merge consecutive urgent windows into alert events
        starts = np.flatnonzero(urgent & ~np.r_[False, urgent[:-1]])
        for s in starts:
            e = s
            while e + 1 < len(urgent) and urgent[e + 1]:
                e += 1
            if y[max(0, s - k + 1): e + 1].sum() == 0:
                fa += 1
        # anomaly episodes (runs of label==1)
        lstarts = np.flatnonzero((y == 1) & ~np.r_[False, (y == 1)[:-1]])
        for s in lstarts:
            e = s
            while e + 1 < len(y) and y[e + 1] == 1:
                e += 1
            episodes += 1
            caught += int(urgent[s: e + k].any())
    return {"false_alerts_per_hour": fa / max(hours, 1e-9),
            "episodes": episodes, "episode_recall": caught / max(episodes, 1)}


# CardioGuard Phase 2: personalized rhythm-anomaly model (MIT-BIH)

**Question:** does scoring each new window against *the user's own baseline* flag arrhythmia episodes better than a generic 60-100 bpm rule or a one-size-fits-all population model?

**Setup**
- Input signal: beat times only (RR intervals), the same thing a watch gives the phone. No ECG morphology.
- Windows: 30 s. Window label = 1 if it holds >= 2 ectopic beats (cardiologist annotations, used for *labels only*).
- Each record plays one "user": first 10 min = calibration baseline, the rest is scored.
- Leave-one-record-out: the model never sees the held-out person's labels.

Set env var `CG_SOURCE=synthetic` to run offline on fake data (pipeline test only, not results).

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

SOURCE = os.environ.get("CG_SOURCE", "mitdb")
N_BASE = 20   # 20 windows x 30 s = 10 min calibration
print("source:", SOURCE)

In [ ]:
raw = load_mitdb("data/mitdb") if SOURCE == "mitdb" else synthetic_records(20, 30)
records = {k: window_features(t, s) for k, (t, s) in raw.items()}
records = {k: v for k, v in records.items() if len(v) > N_BASE + 5}
tot = pd.concat(records.values())
print(f"{len(records)} records, {len(tot)} windows, {tot['label'].mean():.1%} anomalous")

## Leave-one-record-out evaluation

In [ ]:
results = leave_one_record_out(records, n_base=N_BASE)
pooled, per_record = summarize(results)
pooled.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for d in DETECTORS:
    fpr, tpr, _ = roc_curve(results["label"], results[d])
    ax.plot(fpr, tpr, label=f"{d} (AUC {pooled.loc[d,'AUROC']:.2f})")
ax.plot([0,1],[0,1],"k:", lw=1); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("Pooled ROC, held-out records"); ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

### Per-record AUROC (does it work for everyone, or just on average?)

In [ ]:
per_record.round(2).sort_values("personalized_if")

In [ ]:
wins = (per_record["personalized_if"] > per_record["fixed_range"]).mean()
print(f"personalized_if beats fixed_range on {wins:.0%} of records; "
      f"beats population_if on {(per_record['personalized_if'] > per_record['population_if']).mean():.0%}")

## Two-tier alerting
"Worth noting" = any window over a threshold (in-app only). "Urgent" = **2+ consecutive** windows over the threshold. Threshold here = 99th percentile of normal-window scores, chosen on the evaluation data, so treat these numbers as optimistic until validated on a separate split.

In [ ]:
rows = []
for d in DETECTORS:
    thr = results.loc[results.label == 0, d].quantile(0.99)
    rows.append({"detector": d, **sustained_alerts(results, d, thr, k=2)})
pd.DataFrame(rows).set_index("detector").round(3)

## Caveats to state in the writeup
- Labels are ectopic-beat counts from annotations, not "cardiac events"; this is rhythm-anomaly flagging, not diagnosis.
- Only 44-48 people, 30 min each, mostly clinical patients. Baselines are short (10 min), not the 1-2 weeks the app will use.
- Thresholds/tiering tuned on evaluation data; report a held-out split before quoting numbers.
- MIT-BIH beat times come from annotations; a wrist device will have noisier beat detection (motion artifact).